# M11 | Fine-tuning de BERT-mini

Entraîner un modèle Transformer pour prédire les huit catégories
d'un exercice de programmation à partir de deux sources :
- L'énoncé du problème.
- Le code source de sa solution.

Un même encodeur BERT-mini traite séparément les deux sources.
Leurs représentations sont ensuite concaténées et transmises à une
tête de classification multilabel.

Pour limiter le coût de calcul, l'entraînement concerne uniquement la dernière
couche de BERT et la tête de classification.

Trois learning rates pour la dernière couche de BERT sont comparés sur le jeu de validation (1e-5, 2e-5 et 5e-5), afin d'évaluer l'influence de l'intensité du fine-tuning sur les performances du modèle. Un early stopping limite les entraînements inutiles.

Enfin, les seuils de décision sont ajustés pour chaque catégorie
et le meilleur modèle est sauvegardé.

La métrique principale est F1 macro pour s'assurer d'accorder la même importance
à chacune des huit catégories.

## 1. Chargement des données

Utilisation les ensembles d'entraînement et de validation déjà
préparés. Les huit catégories sont représentées par des variables binaires. Les identifiants des exercices et les annotations ne sont jamais transmis au modèle comme variables explicatives, et le jeu de test reste réservé à l'évaluation finale.

In [9]:
import gc
import html
import json
import random
import unicodedata

from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import DataLoader

from transformers import BertConfig, BertModel, BertTokenizer
from huggingface_hub import hf_hub_download

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    hamming_loss,
    precision_recall_fscore_support,
)

from IPython.display import display


racine = next(
    (
        p for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "data" / "entrainement.jsonl").is_file()
    ),
    None
)

if racine is None:
    raise FileNotFoundError(
        "Le dossier data/ est introuvable depuis le répertoire courant."
    )


graine = 42
longueur_max = 512
taille_lot = 16

tags_cibles = [
    "math", "graphs", "strings", "number theory",
    "trees", "geometry", "games", "probabilities"
]

colonnes_cibles = [
    "cible_" + tag.replace(" ", "_")
    for tag in tags_cibles
]

champs_texte = {
    "prob_desc_description": "DESCRIPTION",
    "prob_desc_input_spec": "ENTREES",
    "prob_desc_output_spec": "SORTIES",
    "prob_desc_notes": "NOTES",
}

champ_code = "source_code"


def fixer_graines():
    random.seed(graine)
    np.random.seed(graine)
    torch.manual_seed(graine)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(graine)

    if torch.backends.mps.is_available():
        torch.mps.manual_seed(graine)


fixer_graines()

appareil_entrainement = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)


def charger_ensemble(nom):
    chemin = racine / "data" / f"{nom}.jsonl"

    with chemin.open("r", encoding="utf-8") as fichier:
        return pd.DataFrame([
            json.loads(ligne)
            for ligne in fichier
            if ligne.strip()
        ])


entrainement = charger_ensemble("entrainement")
validation = charger_ensemble("validation")

for donnees in [entrainement, validation]:
    assert set(colonnes_cibles).issubset(donnees.columns)
    assert set(champs_texte).issubset(donnees.columns)
    assert champ_code in donnees.columns
    assert donnees[colonnes_cibles].isin([0, 1]).all().all()

for identifiant in ["fichier_source", "src_uid", "code_uid"]:
    if identifiant in entrainement.columns and identifiant in validation.columns:
        assert set(entrainement[identifiant]).isdisjoint(
            validation[identifiant]
        ), f"Recouvrement détecté : {identifiant}"

y_train = entrainement[colonnes_cibles].to_numpy(dtype=np.int64)
y_validation = validation[colonnes_cibles].to_numpy(dtype=np.int64)

print(f"Entraînement : {len(entrainement)} exercices")
print(f"Validation    : {len(validation)} exercices")
print(f"Catégories    : {len(tags_cibles)}")
print(f"Appareil      : {appareil_entrainement}")

Entraînement : 3470 exercices
Validation    : 739 exercices
Catégories    : 8
Appareil      : mps


## 2. Préparation des entrées

Chaque exercice possède deux entrées :

1. Énoncé : description, spécifications d'entrée et de sortie,
   et notes éventuelles.
2. Code : solution associée à l'exercice.

Les deux sources sont tokenisées séparément avec le tokenizer
de BERT-mini.

Retenir une longueur maximale de 512 tokens par source.
Les séquences plus longues sont tronquées, en conservant le
token spécial `[SEP]` final. Le padding est effectué dynamiquement pour chaque batch afin
de limiter les calculs inutiles.

In [10]:
def construire_enonce(exercice, champs=champs_texte):
    sections = []

    for champ, titre in champs.items():
        valeur = exercice.get(champ)

        if isinstance(valeur, str) and valeur.strip():
            contenu = unicodedata.normalize(
                "NFC", html.unescape(valeur)
            ).strip()

            sections.append(f"{titre}\n{contenu}")

    return "\n\n".join(sections)


nom_encodeur = "prajjwal1/bert-mini"
revision_encodeur = "5e123abc2480f0c4b4cac186d3b3f09299c258fc"

chemin_vocabulaire = hf_hub_download(
    nom_encodeur,
    "vocab.txt",
    revision=revision_encodeur
)

vocabulaire = {
    mot: indice
    for indice, mot in enumerate(
        Path(chemin_vocabulaire)
        .read_text(encoding="utf-8")
        .splitlines()
    )
}

tokenizer = BertTokenizer(
    vocab=vocabulaire,
    do_lower_case=True,
    model_max_length=512
)


def tronquer_ids(ids, limite=longueur_max):
    if len(ids) <= limite:
        return ids

    return ids[:limite - 1] + [ids[-1]]


def preparer_exemples(donnees, cibles):
    enonces = donnees.apply(construire_enonce, axis=1).tolist()
    codes = donnees[champ_code].tolist()

    if any(not texte.strip() for texte in enonces):
        raise ValueError("Certains énoncés sont vides.")

    if any(not isinstance(code, str) or not code.strip() for code in codes):
        raise ValueError("Certains codes sources sont vides.")

    ids_enonces = tokenizer(
        enonces,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False
    )["input_ids"]

    ids_codes = tokenizer(
        codes,
        add_special_tokens=True,
        truncation=False,
        padding=False,
        return_attention_mask=False,
        return_token_type_ids=False,
        verbose=False
    )["input_ids"]

    return [
        {
            "enonce": tronquer_ids(ids_enonce),
            "code": tronquer_ids(ids_code),
            "cibles": torch.tensor(cible, dtype=torch.float32)
        }
        for ids_enonce, ids_code, cible in zip(
            ids_enonces, ids_codes, cibles
        )
    ]


exemples_train = preparer_exemples(entrainement, y_train)
exemples_validation = preparer_exemples(validation, y_validation)


def assembler_lot(exemples):
    lot = {}

    for source in ["enonce", "code"]:
        lot[source] = tokenizer.pad(
            [{"input_ids": exemple[source]} for exemple in exemples],
            padding=True,
            return_tensors="pt"
        )

    lot["cibles"] = torch.stack([
        exemple["cibles"] for exemple in exemples
    ])

    return lot


def creer_chargeur(exemples, melanger=False):
    generateur = torch.Generator().manual_seed(graine)

    return DataLoader(
        exemples,
        batch_size=taille_lot,
        shuffle=melanger,
        collate_fn=assembler_lot,
        num_workers=0,
        generator=generateur
    )


chargeur_validation = creer_chargeur(exemples_validation)

print(f"Exercices tokenisés : {len(exemples_train) + len(exemples_validation)}")
print(f"Longueur maximale   : {longueur_max} tokens par source")
print(f"Batch size          : {taille_lot}")

Exercices tokenisés : 4209
Longueur maximale   : 512 tokens par source
Batch size          : 16


## 3. Architecture du modèle

Utilisation de `prajjwal1/bert-mini`, un Transformer préentraîné
comportant quatre couches et une dimension cachée de 256. Le même encodeur traite successivement l'énoncé et le code. 

Les deux vecteurs de 256 dimensions sont concaténés en un
vecteur de 512 dimensions. Une couche de dropout, suivie d'une couche linéaire, produit
huit logits, un par catégorie.

C'est un fine-tuning partiel, pour lequel les embeddings et les trois premières couches de BERT sont gelés, alors que la dernière couche et la tête de classification sont entraînées. La fonction de perte est `BCEWithLogitsLoss`, adaptée à la classification multilabel.

In [11]:
class ClassifieurBertMini(nn.Module):

    def __init__(self, configuration_bert, dropout=0.1):
        super().__init__()

        self.bert = BertModel(
            configuration_bert,
            add_pooling_layer=False
        )

        self.dropout = nn.Dropout(dropout)

        self.tete = nn.Linear(
            2 * configuration_bert.hidden_size,
            len(tags_cibles)
        )

        self.bert.requires_grad_(False)
        self.bert.encoder.layer[-1].requires_grad_(True)

    def train(self, mode=True):
        super().train(mode)

        self.bert.embeddings.eval()

        for couche in self.bert.encoder.layer[:-1]:
            couche.eval()

        return self

    def forward(self, enonce, code):
        vecteur_enonce = self.bert(
            **enonce
        ).last_hidden_state[:, 0]

        vecteur_code = self.bert(
            **code
        ).last_hidden_state[:, 0]

        fusion = torch.cat(
            [vecteur_enonce, vecteur_code],
            dim=1
        )

        return self.tete(self.dropout(fusion))


fixer_graines()

bert_initial = BertModel.from_pretrained(
    nom_encodeur,
    revision=revision_encodeur,
    add_pooling_layer=False,
    attn_implementation="eager"
)

configuration_bert = bert_initial.config.to_dict()

configuration = BertConfig.from_dict(configuration_bert)
configuration._attn_implementation = "eager"

modele_initial = ClassifieurBertMini(configuration)

modele_initial.bert.load_state_dict(
    bert_initial.state_dict(),
    strict=True
)

etat_initial = {
    cle: valeur.detach().cpu().clone()
    for cle, valeur in modele_initial.state_dict().items()
}

parametres_entrainables = sum(
    p.numel()
    for p in modele_initial.parameters()
    if p.requires_grad
)

parametres_totaux = sum(
    p.numel()
    for p in modele_initial.parameters()
)

print(f"Paramètres totaux       : {parametres_totaux:,}")
print(f"Paramètres entraînables : {parametres_entrainables:,}")
print(f"Proportion entraînable  : {100 * parametres_entrainables / parametres_totaux:.2f} %")

del bert_initial, modele_initial
gc.collect()


def creer_modele():
    fixer_graines()

    config = BertConfig.from_dict(configuration_bert)
    config._attn_implementation = "eager"

    modele = ClassifieurBertMini(config)

    modele.load_state_dict(
        etat_initial,
        strict=True
    )

    return modele.to(appareil_entrainement)

Loading weights: 100%|██████████| 69/69 [00:00<00:00, 42887.82it/s]
[transformers] BertModel LOAD REPORT from: prajjwal1/bert-mini
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from differen

Paramètres totaux       : 11,108,872
Paramètres entraînables : 793,864
Proportion entraînable  : 7.15 %


## 4. Configuration de l'entraînement

Comparaison de trois learning rates pour la dernière couche de BERT :

- `1e-5` : adaptation progressive des poids préentraînés.
- `2e-5` : adaptation intermédiaire.
- `5e-5` : adaptation plus rapide.

Le learning rate de la tête de classification reste fixé à `1e-3`,
afin d'isoler l'effet du learning rate de BERT. Chaque configuration repart des mêmes poids initiaux et peut être entraînée pendant 10 époques au maximum.

À la fin de chaque époque, le modèle est évalué sur le jeu de validation
avec un seuil provisoire de 0,5. L'entraînement s'arrête après une époque sans amélioration suffisante du F1 macro (`min_delta = 0.01`). Le meilleur checkpoint est toujours conservé, et les seuils propres aux huit catégories seront optimisés uniquement après la sélection du modèle.

In [12]:
configurations = [
    {"lr_bert": 1e-5, "lr_tete": 1e-3},
    {"lr_bert": 2e-5, "lr_tete": 1e-3},
    {"lr_bert": 5e-5, "lr_tete": 1e-3},
]

nombre_epoques_max = 10
patience = 1
min_delta = 0.01


def creer_optimiseur(modele, configuration):
    return torch.optim.AdamW(
        [
            {
                "params": modele.bert.encoder.layer[-1].parameters(),
                "lr": configuration["lr_bert"]
            },
            {
                "params": modele.tete.parameters(),
                "lr": configuration["lr_tete"]
            }
        ],
        weight_decay=0.01,
        foreach=False
    )


def deplacer_lot(lot):
    entrees = {
        source: {
            cle: valeur.to(appareil_entrainement)
            for cle, valeur in lot[source].items()
        }
        for source in ["enonce", "code"]
    }

    cibles = lot["cibles"].to(appareil_entrainement)

    return entrees, cibles


def synchroniser():
    if appareil_entrainement == "mps":
        torch.mps.synchronize()
    elif appareil_entrainement == "cuda":
        torch.cuda.synchronize()


def etape_apprentissage(modele, optimiseur, lot):
    modele.train()

    entrees, cibles = deplacer_lot(lot)

    optimiseur.zero_grad(set_to_none=True)

    logits = modele(**entrees)

    perte = nn.functional.binary_cross_entropy_with_logits(
        logits, cibles
    )

    if not torch.isfinite(perte).item():
        raise RuntimeError("La perte d'entraînement n'est pas finie.")

    perte.backward()

    nn.utils.clip_grad_norm_(
        [p for p in modele.parameters() if p.requires_grad],
        max_norm=1.0,
        foreach=False
    )

    optimiseur.step()

    return float(perte.detach().cpu())


def calculer_scores(modele, chargeur):
    modele.eval()

    scores = []

    with torch.inference_mode():
        for lot in chargeur:
            entrees, _ = deplacer_lot(lot)

            probabilites = torch.sigmoid(
                modele(**entrees)
            )

            scores.append(
                probabilites.cpu().numpy()
            )

    return np.concatenate(scores, axis=0)


def evaluer_predictions(reels, predictions):
    return {
        "f1_macro": float(
            f1_score(
                reels, predictions,
                average="macro",
                zero_division=0
            )
        ),
        "f1_micro": float(
            f1_score(
                reels, predictions,
                average="micro",
                zero_division=0
            )
        ),
        "exact_match_ratio": float(
            accuracy_score(reels, predictions)
        ),
        "hamming_loss": float(
            hamming_loss(reels, predictions)
        )
    }

## 5. Fine-tuning et sélection du modèle

Entraînement indépendant des trois configurations, correspondant aux learning rates BERT `1e-5`, `2e-5` et `5e-5`. Chaque configuration repart des mêmes poids préentraînés.

À chaque époque :

1. Le modèle apprend sur les batches d'entraînement.
2. Les probabilités sont calculées sur la validation.
3. Le F1 macro est mesuré avec un seuil de 0,5.
4. Le meilleur checkpoint est conservé.
5. L'early stopping décide de poursuivre ou d'arrêter.

La configuration ayant obtenu le meilleur F1 macro de validation est retenue pour la suite. Cette comparaison permet d'étudier l'influence du learning rate sur la convergence et les performances du fine-tuning.

In [13]:
def entrainer_configuration(configuration):
    modele = creer_modele()
    optimiseur = creer_optimiseur(modele, configuration)

    chargeur_train = creer_chargeur(
        exemples_train,
        melanger=True
    )

    meilleur_f1 = -1.0
    meilleur_etat = None
    meilleure_epoque = None

    reference_progres = -1.0
    sans_progres = 0

    historique = []

    synchroniser()
    debut_configuration = perf_counter()

    for epoque in range(1, nombre_epoques_max + 1):
        debut_epoque = perf_counter()
        somme_pertes = 0.0

        for lot in chargeur_train:
            perte = etape_apprentissage(
                modele, optimiseur, lot
            )

            somme_pertes += perte * len(lot["cibles"])

        scores = calculer_scores(
            modele, chargeur_validation
        )

        metriques = evaluer_predictions(
            y_validation,
            scores >= 0.5
        )

        synchroniser()

        f1_actuel = metriques["f1_macro"]

        historique.append({
            "epoque": epoque,
            "perte_train": somme_pertes / len(entrainement),
            "duree_s": perf_counter() - debut_epoque,
            **metriques
        })

        print(
            f"Époque {epoque:02d} | "
            f"Loss = {historique[-1]['perte_train']:.4f} | "
            f"F1 macro = {f1_actuel:.4f}",
            flush=True
        )

        if f1_actuel > meilleur_f1:
            meilleur_f1 = f1_actuel
            meilleure_epoque = epoque

            meilleur_etat = {
                cle: valeur.detach().cpu().clone()
                for cle, valeur in modele.state_dict().items()
            }

        if f1_actuel > reference_progres + min_delta:
            reference_progres = f1_actuel
            sans_progres = 0
        else:
            sans_progres += 1

        if sans_progres >= patience:
            print(
                f"Early stopping à l'époque {epoque}. "
                f"Meilleure époque : {meilleure_epoque}."
            )
            break

    duree = perf_counter() - debut_configuration

    modele.load_state_dict(
        meilleur_etat,
        strict=True
    )

    scores = calculer_scores(
        modele, chargeur_validation
    )

    resultat = {
        **configuration,
        "meilleure_epoque": meilleure_epoque,
        "epoques_executees": len(historique),
        "temps_entrainement_s": duree,
        **evaluer_predictions(
            y_validation,
            scores >= 0.5
        )
    }

    del modele, optimiseur
    gc.collect()

    if appareil_entrainement == "cuda":
        torch.cuda.empty_cache()

    return resultat, meilleur_etat, scores, historique


experiences = []
historiques = {}

configuration_retenue = None
etat_retenu = None
scores_validation = None

for config in configurations:
    print(
        f"\n=== LR BERT : {config['lr_bert']} ===",
        flush=True
    )

    resultat, etat, scores, historique = (
        entrainer_configuration(config)
    )

    experiences.append(resultat)
    historiques[str(config["lr_bert"])] = historique

    if (
        configuration_retenue is None
        or resultat["f1_macro"] > configuration_retenue["f1_macro"]
    ):
        configuration_retenue = resultat
        etat_retenu = etat
        scores_validation = scores


recherche = (
    pd.DataFrame(experiences)
    .sort_values(
        "f1_macro",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nComparaison des configurations :")
display(recherche.round(4))

print("\nHistorique de la meilleure configuration :")
display(
    pd.DataFrame(
        historiques[str(configuration_retenue["lr_bert"])]
    ).round(4)
)


=== LR BERT : 1e-05 ===
Époque 01 | Loss = 0.2558 | F1 macro = 0.0480
Époque 02 | Loss = 0.2130 | F1 macro = 0.1415
Époque 03 | Loss = 0.1942 | F1 macro = 0.2428
Époque 04 | Loss = 0.1842 | F1 macro = 0.3138
Époque 05 | Loss = 0.1763 | F1 macro = 0.3791
Époque 06 | Loss = 0.1709 | F1 macro = 0.3361
Early stopping à l'époque 6. Meilleure époque : 5.

=== LR BERT : 2e-05 ===
Époque 01 | Loss = 0.2540 | F1 macro = 0.0640
Époque 02 | Loss = 0.2061 | F1 macro = 0.2076
Époque 03 | Loss = 0.1859 | F1 macro = 0.3134
Époque 04 | Loss = 0.1758 | F1 macro = 0.3769
Époque 05 | Loss = 0.1686 | F1 macro = 0.4409
Époque 06 | Loss = 0.1625 | F1 macro = 0.3528
Early stopping à l'époque 6. Meilleure époque : 5.

=== LR BERT : 5e-05 ===
Époque 01 | Loss = 0.2490 | F1 macro = 0.1149
Époque 02 | Loss = 0.1955 | F1 macro = 0.2469
Époque 03 | Loss = 0.1750 | F1 macro = 0.4226
Époque 04 | Loss = 0.1645 | F1 macro = 0.5221
Époque 05 | Loss = 0.1568 | F1 macro = 0.4924
Early stopping à l'époque 5. Meilleure ép

,lr_bert,lr_tete,meilleure_epoque,epoques_executees,temps_entrainement_s,f1_macro,f1_micro,exact_match_ratio,hamming_loss
0,0.0,0.001,4,5,436.8245,0.5221,0.5576,0.5250,0.0695
1,0.0,0.001,5,6,503.3689,0.4409,0.4579,0.5413,0.0697
2,0.0,0.001,5,6,464.0076,0.3791,0.4324,0.5372,0.0710



Historique de la meilleure configuration :


,epoque,perte_train,duree_s,f1_macro,f1_micro,exact_match_ratio,hamming_loss
0,1,0.2490,86.0932,0.1149,0.2952,0.4641,0.0824
1,2,0.1955,86.4559,0.2469,0.3830,0.5210,0.0736
2,3,0.1750,88.6256,0.4226,0.4444,0.5453,0.0702
3,4,0.1645,88.1037,0.5221,0.5576,0.5250,0.0695
4,5,0.1568,87.4221,0.4924,0.4796,0.5629,0.0668


## 6. Optimisation des seuils de classification

Le modèle produit une probabilité pour chacune des huit catégories. Pour chaque tag, le seuil maximisant son F1 sur le jeu de validation est recherché. Les candidats sont obtenus à partir des quantiles des probabilités prédites, en ajoutant systématiquement le seuil de 0,5. Cette optimisation ne modifie donc pas les poids du modèle.

In [14]:
seuil_defaut = 0.5

quantiles_seuils = [
    0.50, 0.70, 0.80, 0.85,
    0.90, 0.925, 0.95, 0.965,
    0.975, 0.985, 0.99, 0.995
]

seuils = []

for indice, tag in enumerate(tags_cibles):
    probabilites = scores_validation[:, indice]
    cibles = y_validation[:, indice]

    candidats = np.unique(
        np.append(
            np.quantile(probabilites, quantiles_seuils),
            seuil_defaut
        )
    )

    evaluations = [
        (
            f1_score(
                cibles,
                probabilites >= seuil,
                zero_division=0
            ),
            float(seuil)
        )
        for seuil in candidats
    ]

    meilleur = min(
        evaluations,
        key=lambda resultat: (
            -resultat[0],
            abs(resultat[1] - seuil_defaut),
            resultat[1]
        )
    )

    seuils.append(meilleur[1])

seuils = np.array(seuils)

tableau_seuils = pd.DataFrame({
    "tag": tags_cibles,
    "seuil": seuils
})

display(tableau_seuils.round(4))

,tag,seuil
0,math,0.5000
1,graphs,0.1337
2,strings,0.1405
3,number theory,0.5000
4,trees,0.3772
5,geometry,0.5000
6,games,0.3719
7,probabilities,0.1122


## 7. Résultats sur le jeu de validation

Comparaison des performances du modèle retenu avant et après l'optimisation des seuils. Les métriques globales sont :

- **F1 macro :** moyenne des F1 des huit catégories.
- **F1 micro :** F1 calculé sur l'ensemble des décisions binaires.
- **Exact Match Ratio :** proportion d'exercices pour lesquels
  toutes les étiquettes sont correctement prédites.
- **Hamming Loss :** proportion de décisions binaires incorrectes.

La précision, le rappel, le F1 et le support de chaque catégorie sont aussi calculés sur l'ensemble de validation.

In [15]:
predictions_defaut = (
    scores_validation >= seuil_defaut
).astype(int)

predictions_ajustees = (
    scores_validation >= seuils
).astype(int)


bilan = pd.DataFrame([
    {
        "seuils": "Défaut (0.5)",
        **evaluer_predictions(
            y_validation,
            predictions_defaut
        )
    },
    {
        "seuils": "Optimisés",
        **evaluer_predictions(
            y_validation,
            predictions_ajustees
        )
    }
])

print("Performances globales :")
display(bilan.round(4))


precisions, rappels, f1, supports = (
    precision_recall_fscore_support(
        y_validation,
        predictions_ajustees,
        average=None,
        zero_division=0
    )
)

par_tag = pd.DataFrame({
    "tag": tags_cibles,
    "support": supports.astype(int),
    "seuil": seuils,
    "precision": precisions,
    "rappel": rappels,
    "f1": f1
})

print("\nPerformances par catégorie :")
display(par_tag.round(4))

Performances globales :


,seuils,f1_macro,f1_micro,exact_match_ratio,hamming_loss
0,Défaut (0.5),0.5221,0.5576,0.5250,0.0695
1,Optimisés,0.5839,0.5913,0.5129,0.0734



Performances par catégorie :


,tag,support,seuil,precision,rappel,f1
0,math,211,0.5000,0.5317,0.6351,0.5788
1,graphs,81,0.1337,0.5315,0.7284,0.6146
2,strings,63,0.1405,0.5811,0.6825,0.6277
3,number theory,52,0.5000,0.6250,0.3846,0.4762
4,trees,49,0.3772,0.8378,0.6327,0.7209
5,geometry,25,0.5000,0.7857,0.4400,0.5641
6,games,16,0.3719,0.9167,0.6875,0.7857
7,probabilities,14,0.1122,0.2632,0.3571,0.3030


## 8. Sauvegarde du modèle retenu

Sauvegarde des éléments nécessaires pour reproduire les prédictions :

- Les poids du meilleur modèle.
- La configuration de l'architecture BERT-mini.
- Le tokenizer.
- Les huit seuils de classification.
- Les paramètres d'entraînement et les résultats de validation.

In [17]:
dossier_modele = (
    racine / "models" / "m11_bert_mini_finetuning"
)

dossier_modele.mkdir(
    parents=True,
    exist_ok=True
)

torch.save(
    etat_retenu,
    dossier_modele / "modele.pt"
)

tokenizer.save_pretrained(
    dossier_modele / "tokenizer"
)

informations = {
    "modele": "M11",
    "nom_encodeur": nom_encodeur,
    "revision_encodeur": revision_encodeur,
    "configuration_bert": configuration_bert,

    "architecture": {
        "pooling": "CLS",
        "fusion": "concatenation_enonce_code",
        "dropout": 0.1,
        "couches_bert_entrainees": 1,
        "attention": "eager"
    },

    "tags": tags_cibles,
    "champs_texte": champs_texte,
    "champ_code": champ_code,

    "tokenisation": {
        "longueur_max": longueur_max,
        "troncature": "debut_avec_SEP_final",
        "padding": "dynamique"
    },

    "seuils": seuils.tolist(),

    "entrainement": {
        "graine": graine,
        "batch_size": taille_lot,
        "nombre_epoques_max": nombre_epoques_max,
        "patience": patience,
        "min_delta": min_delta,
        "weight_decay": 0.01,
        "clip_grad_norm": 1.0,
        "loss": "BCEWithLogitsLoss",
        "configuration_retenue": configuration_retenue
    }
}

(dossier_modele / "configuration.json").write_text(
    json.dumps(
        informations,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)

recherche.to_csv(
    dossier_modele / "recherche_hyperparametres.csv",
    index=False
)

bilan.to_csv(
    dossier_modele / "metriques_validation.csv",
    index=False
)

par_tag.to_csv(
    dossier_modele / "metriques_par_tag_validation.csv",
    index=False
)

print("Modèle et résultats sauvegardés.")

Modèle et résultats sauvegardés.
